In [16]:
import pandas as pd
import numpy as np
import json
import joblib
import shap
import shap
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix
from statsmodels.stats.contingency_tables import mcnemar

In [17]:
os.makedirs('dashboard_data', exist_ok=True)

data_path = r'C:\Users\adity\Desktop\Kepler-Exoplanet Project\data'
df = pd.read_csv(os.path.join(data_path, 'kepler_clean.csv'))

In [18]:
TARGET = 'koi_disposition'
df_model = df[df[TARGET].isin(['CONFIRMED', 'FALSE POSITIVE'])].copy()

KEY_FEATURES = [
    'koi_period', 'koi_prad', 'koi_teq', 'koi_insol',
    'koi_model_snr', 'koi_steff', 'koi_slogg',
    'koi_srad', 'koi_score', 'koi_depth',
    'koi_duration', 'koi_impact',
    'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec'
]

X = df_model[KEY_FEATURES]
y_raw = df_model[TARGET]
le = LabelEncoder()
y = le.fit_transform(y_raw)

imputer = SimpleImputer(strategy='median')

df_model['star_id'] = df_model['ra'].round(5).astype(str) + '_' + df_model['dec'].round(5).astype(str)
groups = df_model['star_id']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_imp, y_train)

FEATURE_TIERS = {
    'Full': KEY_FEATURES,
    'No koi_score': [f for f in KEY_FEATURES if f != 'koi_score'],
    'Raw Physical': [f for f in KEY_FEATURES if f not in 
                                ['koi_score', 'koi_fpflag_nt', 'koi_fpflag_ss', 'koi_fpflag_co', 'koi_fpflag_ec']]
}

meta_cols = ['koi_period', 'koi_prad', 'koi_depth', 'koi_impact', 'koi_score', TARGET]
df_model.iloc[test_idx][meta_cols].reset_index(drop=True).to_csv('dashboard_data/test_metadata.csv', index=False)
np.save('dashboard_data/y_test.npy', y_test)

print(f"Setup done, Train: {len(X_train):,} | Test: {len(X_test):,}")

Setup done, Train: 4,941 | Test: 1,262


In [19]:
def make_models():
    return {
        'Random Forest': RandomForestClassifier(n_estimators=200, min_samples_split=5, random_state=42, n_jobs=-1),
        'XGBoost': XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
                                colsample_bytree=0.8, use_label_encoder=False, eval_metric='logloss', random_state=42),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
        'SVM': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'LightGBM': LGBMClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
                                colsample_bytree=0.8, random_state=42, verbose=-1)
    }

all_metrics = {}
all_predictions = {}

for tier_name, features in FEATURE_TIERS.items():
    X_train_tier = pd.DataFrame(X_train_bal, columns=KEY_FEATURES)[features]
    X_test_tier = pd.DataFrame(X_test_imp, columns=KEY_FEATURES)[features]
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_tier)
    X_test_scaled = scaler.transform(X_test_tier)

    all_metrics[tier_name] = {}
    all_predictions[tier_name] = {}

    for name, model in make_models().items():
        needs_scaling = name in ('SVM', 'Logistic Regression')
        if needs_scaling:
            model.fit(X_train_scaled, y_train_bal)
            y_pred = model.predict(X_test_scaled)
            y_prob = model.predict_proba(X_test_scaled)[:, 1]
        else:
            model.fit(X_train_tier, y_train_bal)
            y_pred = model.predict(X_test_tier)
            y_prob = model.predict_proba(X_test_tier)[:, 1]

        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        cm = confusion_matrix(y_test, y_pred).tolist()

        all_metrics[tier_name][name] = {'f1': f1, 'auc': auc, 'confusion_matrix': cm}
        all_predictions[tier_name][name] = {'y_pred': y_pred.tolist(), 'y_prob': y_prob.tolist()}

        print(f"{tier_name} | {name:<20} F1={f1:.4f} AUC={auc:.4f}")

    if tier_name == 'Raw Physical':
        joblib.dump(make_models()['XGBoost'].fit(X_train_tier, y_train_bal), 'dashboard_data/xgb_raw_model.pkl')
        X_test_tier.to_csv('dashboard_data/X_test_raw.csv', index=False)
    if tier_name == 'Full':
        joblib.dump(make_models()['XGBoost'].fit(X_train_tier, y_train_bal), 'dashboard_data/xgb_full_model.pkl')
        X_test_tier.to_csv('dashboard_data/X_test_full.csv', index=False)

with open('dashboard_data/metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
with open('dashboard_data/predictions.json', 'w') as f:
    json.dump(all_predictions, f, indent=2)

print("\nSaved metrics.json and predictions.json")

Full | Random Forest        F1=0.9941 AUC=0.9992


c:\Users\adity\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:15:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Full | XGBoost              F1=0.9948 AUC=0.9995
Full | Gradient Boosting    F1=0.9935 AUC=0.9995


c:\Users\adity\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Full | SVM                  F1=0.9961 AUC=0.9985
Full | Logistic Regression  F1=0.9961 AUC=0.9987
Full | LightGBM             F1=0.9941 AUC=0.9996


...

Raw Physical | XGBoost              F1=0.9412 AUC=0.9786
Raw Physical | Gradient Boosting    F1=0.9291 AUC=0.9749


c:\Users\adity\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Raw Physical | SVM                  F1=0.8875 AUC=0.9535
Raw Physical | Logistic Regression  F1=0.8622 AUC=0.9280
Raw Physical | LightGBM             F1=0.9382 AUC=0.9781


c:\Users\adity\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:16:11] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Saved metrics.json and predictions.json


In [20]:
xgb_full = joblib.load('dashboard_data/xgb_full_model.pkl')
X_test_full = pd.read_csv('dashboard_data/X_test_full.csv')
explainer_full = shap.TreeExplainer(xgb_full)
shap_values_full = explainer_full.shap_values(X_test_full)
np.save('dashboard_data/shap_values_full.npy', shap_values_full)


xgb_raw = joblib.load('dashboard_data/xgb_raw_model.pkl')
X_test_raw = pd.read_csv('dashboard_data/X_test_raw.csv')
explainer_raw = shap.TreeExplainer(xgb_raw)
shap_values_raw = explainer_raw.shap_values(X_test_raw)
np.save('dashboard_data/shap_values_raw.npy', shap_values_raw)

print("SHAP values for Full and Raw physical tiers.")

SHAP values for Full and Raw physical tiers.
